# Notebook 07 — Clone × Process Interaction Simulation

## Goal

Notebook07 extends the CLD pipeline from clone selection to clone × process decision support.

Earlier notebooks asked:

> Which clone should we advance?

Notebook07 now asks:

> Which clone should we advance under which process condition?

This notebook uses the predicted late-stage outcomes from Notebook03/04 together with clone-specific process-response latent variables generated by the synthetic CLD generator.

## New in Phase 1

This version adds explicit clone × process interaction biology.

Each clone now has hidden process-response axes:

- `nutrient_utilization_efficiency`
- `stress_adaptation_capacity`
- `perfusion_rescue_potential`
- `temperature_shift_responsiveness`
- `feed_responsiveness`
- `secretion_burden_index`
- `process_risk_sensitivity`

These are not used as early-stage ML features.

Instead, they represent hidden clone biology that determines how each clone may respond to process interventions such as:

- balanced feed
- rich media
- nutrient-limited strategy
- perfusion rescue
- temperature shift
- intensified feeding

## Interpretation

Notebook07 should be interpreted as a hypothesis-generating process simulation layer.

It does not claim that the process effects are validated.

Instead, it demonstrates how a future CDMO-facing system could move from:

> clone ranking

to:

> clone × process recommendation

In [1]:
# --------------------------------------------------
# Load prediction table from Notebook03
# and process-response latent truth from generator
# --------------------------------------------------

import pandas as pd
import numpy as np
from pathlib import Path

scenario = "legacy"   # "legacy" or "optimized"
n_clones = 5000

PRED_PATH = Path("../data/synthetic/processed") / (
    f"predictions_03b_qp_{n_clones}_{scenario}.csv"
)

LATENT_PATH = Path("../data/synthetic/raw") / (
    f"clone_latent_truths_{n_clones}_{scenario}.csv"
)

print("Loading prediction table:", PRED_PATH)
df = pd.read_csv(PRED_PATH)

print("Loading clone latent truth:", LATENT_PATH)
latent = pd.read_csv(LATENT_PATH)

process_latent_cols = [
    "clone_id",
    "nutrient_utilization_efficiency",
    "stress_adaptation_capacity",
    "perfusion_rescue_potential",
    "temperature_shift_responsiveness",
    "feed_responsiveness",
    "secretion_burden_index",
    "process_risk_sensitivity",
]

missing_latent = [c for c in process_latent_cols if c not in latent.columns]
print("Missing process-latent columns:", missing_latent)
assert len(missing_latent) == 0, f"Missing process-latent columns: {missing_latent}"

df = df.merge(
    latent[process_latent_cols],
    on="clone_id",
    how="left",
)

required_cols = [
    "clone_id",
    "pred_late_qp",
    "pred_qp_drop",
    "pred_late_agg",
    "pred_stable_prob",
    "pred_rescue_score",
    "pred_rescue_label",
    "true_late_qp",
    "true_qp_drop",
    "true_late_agg",
    "nutrient_utilization_efficiency",
    "stress_adaptation_capacity",
    "perfusion_rescue_potential",
    "temperature_shift_responsiveness",
    "feed_responsiveness",
    "secretion_burden_index",
    "process_risk_sensitivity",
]

missing = [c for c in required_cols if c not in df.columns]
print("Missing required columns:", missing)
assert len(missing) == 0, f"Missing required columns: {missing}"

print("Prediction + latent merged shape:", df.shape)
display(df.head())
display(df[process_latent_cols].describe())

Loading prediction table: ../data/synthetic/processed/predictions_03b_qp_5000_legacy.csv
Loading clone latent truth: ../data/synthetic/raw/clone_latent_truths_5000_legacy.csv
Missing process-latent columns: []
Missing required columns: []
Prediction + latent merged shape: (1000, 27)


,clone_id,pred_qp_drop,pred_late_qp,pred_late_agg,pred_stable_prob,pred_stable_label,true_qp_drop,true_late_qp,true_late_agg,true_stable_label,...,rescue_not_already_pass,pred_rescue_score,pred_rescue_label,nutrient_utilization_efficiency,stress_adaptation_capacity,perfusion_rescue_potential,temperature_shift_responsiveness,feed_responsiveness,secretion_burden_index,process_risk_sensitivity
0,CLONE_1502,0.381373,2.711177e-08,9.583127,0.370940,0,0.111023,4.176362e-08,12.566994,1,...,0.486344,0.598822,1,0.714663,0.643065,0.450092,0.287103,0.453787,0.009955,0.265650
1,CLONE_2587,0.489493,6.445184e-08,3.127400,0.169707,0,0.691891,3.691256e-08,0.918511,0,...,0.769478,0.264869,0,0.741458,0.733727,0.260307,0.588883,0.623808,0.461429,0.340751
2,CLONE_2654,0.362087,3.071979e-08,7.313653,0.492969,0,0.009193,4.002863e-08,6.558486,1,...,0.314649,0.571715,0,0.427308,0.512669,0.470485,0.238474,0.373872,0.227617,0.534961
3,CLONE_1056,0.353752,6.793292e-08,6.868662,0.478848,0,0.708930,3.026369e-08,7.990409,0,...,0.334518,0.549744,0,0.551334,0.527079,0.447419,0.543270,0.735064,0.354122,0.540352
4,CLONE_0706,0.616517,1.729449e-07,8.278383,0.036653,0,0.740570,1.029717e-07,7.441547,0,...,0.956685,0.397218,0,0.417567,0.482788,0.355846,0.673496,0.660623,0.676830,0.496857


,nutrient_utilization_efficiency,stress_adaptation_capacity,perfusion_rescue_potential,temperature_shift_responsiveness,feed_responsiveness,secretion_burden_index,process_risk_sensitivity
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,0.608991,0.642516,0.438471,0.499358,0.491757,0.454408,0.505118
std,0.136374,0.138156,0.141638,0.134336,0.130223,0.221333,0.153076
min,0.157823,0.000000,0.009806,0.139244,0.029765,0.000000,0.000000
25%,0.525684,0.558239,0.335844,0.410332,0.399839,0.293117,0.397109
50%,0.613092,0.642345,0.443031,0.491186,0.495391,0.456173,0.498499
75%,0.699970,0.732393,0.530205,0.594815,0.577970,0.618664,0.609713
max,1.000000,1.000000,0.901515,1.000000,0.883416,1.000000,1.000000


## Step 1 — Utility helpers

We define utility scores for evaluating clone quality.

Higher utility means:

- higher late qP
- lower qP drop
- lower aggregation

The true utility is used only for offline evaluation.

In [2]:
# --------------------------------------------------
# Step 1 — Utility helper functions
# --------------------------------------------------

def z(s):
    s = pd.Series(s).astype(float)
    return (s - s.mean()) / (s.std(ddof=0) + 1e-9)

def z01(s):
    s = pd.Series(s).astype(float)
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

def make_true_utility(df):
    return (
        1.0 * z(df["true_late_qp"])
        - 1.0 * z(df["true_qp_drop"])
        - 0.2 * z(df["true_late_agg"])
    )

def make_pred_utility(df, qp_col, drop_col, agg_col):
    return (
        1.0 * z(df[qp_col])
        - 1.0 * z(df[drop_col])
        - 0.2 * z(df[agg_col])
    )

def topk_overlap_by_id(true_ids, selected_ids, k=10):
    return len(set(true_ids) & set(selected_ids)) / k

df["true_util"] = make_true_utility(df)
df["baseline_score"] = make_pred_utility(
    df,
    qp_col="pred_late_qp",
    drop_col="pred_qp_drop",
    agg_col="pred_late_agg",
)

display(df[[
    "clone_id",
    "pred_late_qp",
    "pred_qp_drop",
    "pred_late_agg",
    "pred_rescue_score",
    "true_late_qp",
    "true_qp_drop",
    "true_late_agg",
    "true_util",
    "baseline_score",
]].head())

,clone_id,pred_late_qp,pred_qp_drop,pred_late_agg,pred_rescue_score,true_late_qp,true_qp_drop,true_late_agg,true_util,baseline_score
0,CLONE_1502,2.711177e-08,0.381373,9.583127,0.598822,4.176362e-08,0.111023,12.566994,1.032173,-0.084743
1,CLONE_2587,6.445184e-08,0.489493,3.127400,0.264869,3.691256e-08,0.691891,0.918511,-1.015188,-0.469116
2,CLONE_2654,3.071979e-08,0.362087,7.313653,0.571715,4.002863e-08,0.009193,6.558486,1.877311,0.304273
3,CLONE_1056,6.793292e-08,0.353752,6.868662,0.549744,3.026369e-08,0.708930,7.990409,-1.526406,0.506410
4,CLONE_0706,1.729449e-07,0.616517,8.278383,0.397218,1.029717e-07,0.740570,7.441547,-1.615101,-1.881235


## Step 2 — Define virtual process conditions

We define several conservative virtual process conditions.

Compared with the previous version, this version includes process conditions that can interact with clone-specific latent biology:

- `baseline`: no intervention
- `rich_media`: productivity-oriented condition, but may increase stress and aggregation
- `nutrient_limited`: conservative condition for stability and quality risk reduction
- `balanced_feed`: moderate productivity and risk-balanced condition
- `mild_temp_shift`: temperature-shift-like condition for secretion burden relief
- `perfusion_like`: media-exchange / waste-removal-like condition for rescue-sensitive clones

These are not real manufacturing recipes.

They are simplified biological process scenarios for testing clone × process interaction logic.

In [3]:
# --------------------------------------------------
# Step 2 — Define virtual process conditions
# --------------------------------------------------

# Values represent base process effect before clone-specific interaction.
#
# qp_effect:
#   relative change in predicted late qP
#
# drop_effect:
#   absolute change in predicted qP drop
#   negative = improved stability
#   positive = worse stability
#
# agg_effect:
#   relative change in predicted aggregation
#   negative = reduced aggregation
#   positive = increased aggregation
#
# stress_load:
#   process burden / risk intensity
#
# main_driver:
#   which clone-specific latent variable should drive the response

process_conditions = {
    "baseline": {
        "qp_effect": 0.00,
        "drop_effect": 0.00,
        "agg_effect": 0.00,
        "stress_load": 0.00,
        "main_driver": "none",
    },
    "rich_media": {
        "qp_effect": 0.08,
        "drop_effect": 0.03,
        "agg_effect": 0.06,
        "stress_load": 0.35,
        "main_driver": "feed_responsiveness",
    },
    "nutrient_limited": {
        "qp_effect": -0.04,
        "drop_effect": -0.08,
        "agg_effect": -0.08,
        "stress_load": -0.20,
        "main_driver": "process_risk_sensitivity",
    },
    "balanced_feed": {
        "qp_effect": 0.04,
        "drop_effect": -0.04,
        "agg_effect": -0.04,
        "stress_load": 0.05,
        "main_driver": "nutrient_utilization_efficiency",
    },
    "mild_temp_shift": {
        "qp_effect": 0.03,
        "drop_effect": -0.05,
        "agg_effect": -0.06,
        "stress_load": -0.10,
        "main_driver": "temperature_shift_responsiveness",
    },
    "perfusion_like": {
        "qp_effect": 0.02,
        "drop_effect": -0.06,
        "agg_effect": -0.05,
        "stress_load": -0.15,
        "main_driver": "perfusion_rescue_potential",
    },
}

process_condition_table = pd.DataFrame(process_conditions).T
display(process_condition_table)

,qp_effect,drop_effect,agg_effect,stress_load,main_driver
baseline,0.0,0.0,0.0,0.0,none
rich_media,0.08,0.03,0.06,0.35,feed_responsiveness
nutrient_limited,-0.04,-0.08,-0.08,-0.2,process_risk_sensitivity
balanced_feed,0.04,-0.04,-0.04,0.05,nutrient_utilization_efficiency
mild_temp_shift,0.03,-0.05,-0.06,-0.1,temperature_shift_responsiveness
perfusion_like,0.02,-0.06,-0.05,-0.15,perfusion_rescue_potential


## Step 3 — Clone-specific process-response sensitivity

This step combines two kinds of information:

1. **Predicted phenotype sensitivity**
   - rescue score
   - predicted qP drop band
   - predicted aggregation band

2. **Generator-derived process-response latent biology**
   - nutrient utilization efficiency
   - stress adaptation capacity
   - perfusion rescue potential
   - temperature shift responsiveness
   - feed responsiveness
   - secretion burden index
   - process risk sensitivity

The goal is not to give the model hidden information for prediction.

Instead, Notebook07 uses these latent variables only inside the process simulation layer.

This represents the idea that some clones are more process-responsive than others.

In [4]:
# --------------------------------------------------
# Step 3 — Clone-specific process-response sensitivity
# --------------------------------------------------

def triangular_band_score(x, low, high, peak=None):
    x = pd.Series(x).astype(float)

    if peak is None:
        peak = (low + high) / 2

    score = pd.Series(np.zeros(len(x)), index=x.index)

    left = (x >= low) & (x <= peak)
    right = (x > peak) & (x <= high)

    score.loc[left] = (x.loc[left] - low) / (peak - low + 1e-12)
    score.loc[right] = (high - x.loc[right]) / (high - peak + 1e-12)

    return score.clip(0, 1)

work = df.copy()

# Prediction-derived sensitivity
work["sens_rescue"] = z01(work["pred_rescue_score"])

work["sens_stability_band"] = triangular_band_score(
    work["pred_qp_drop"],
    low=0.20,
    peak=0.35,
    high=0.60,
)

work["sens_agg_band"] = triangular_band_score(
    work["pred_late_agg"],
    low=5.0,
    peak=9.0,
    high=16.0,
)

# Latent process-response biology
work["latent_process_response"] = (
    0.20 * work["nutrient_utilization_efficiency"]
    + 0.20 * work["stress_adaptation_capacity"]
    + 0.15 * work["perfusion_rescue_potential"]
    + 0.15 * work["temperature_shift_responsiveness"]
    + 0.15 * work["feed_responsiveness"]
    + 0.15 * (1.0 - work["process_risk_sensitivity"])
)

# Overall process sensitivity:
# prediction-derived rescue potential + latent process responsiveness.
work["process_sensitivity"] = (
    0.40 * work["sens_rescue"]
    + 0.20 * work["sens_stability_band"]
    + 0.15 * work["sens_agg_band"]
    + 0.25 * work["latent_process_response"]
)

work["process_sensitivity"] = work["process_sensitivity"].clip(0, 1)

print("=== Process sensitivity summary ===")
display(work["process_sensitivity"].describe())

display(
    work[
        [
            "clone_id",
            "pred_rescue_score",
            "pred_qp_drop",
            "pred_late_agg",
            "sens_rescue",
            "sens_stability_band",
            "sens_agg_band",
            "latent_process_response",
            "process_sensitivity",
            "nutrient_utilization_efficiency",
            "stress_adaptation_capacity",
            "perfusion_rescue_potential",
            "temperature_shift_responsiveness",
            "feed_responsiveness",
            "secretion_burden_index",
            "process_risk_sensitivity",
        ]
    ]
    .sort_values("process_sensitivity", ascending=False)
    .head(15)
)

=== Process sensitivity summary ===


count    1000.000000
mean        0.477627
std         0.124138
min         0.135975
25%         0.406346
50%         0.474080
75%         0.559112
max         0.895472
Name: process_sensitivity, dtype: float64

,clone_id,pred_rescue_score,pred_qp_drop,pred_late_agg,sens_rescue,sens_stability_band,sens_agg_band,latent_process_response,process_sensitivity,nutrient_utilization_efficiency,stress_adaptation_capacity,perfusion_rescue_potential,temperature_shift_responsiveness,feed_responsiveness,secretion_burden_index,process_risk_sensitivity
180,CLONE_4625,1.000000,0.357910,8.834887,1.000000,0.968361,0.958722,0.631968,0.895472,0.718573,0.728852,0.415454,0.638964,0.454035,0.803980,0.225237
425,CLONE_4976,0.659422,0.349117,9.046036,0.659422,0.994116,0.993423,0.558950,0.751343,0.578733,0.893842,0.275287,0.460497,0.385009,0.300869,0.357894
29,CLONE_0487,0.648099,0.356164,9.046934,0.648099,0.975342,0.993295,0.565265,0.744619,0.816690,0.616631,0.375793,0.414309,0.568862,0.341670,0.501628
143,CLONE_3900,0.653145,0.369129,8.845331,0.653145,0.923484,0.961333,0.607691,0.742077,0.781594,0.817875,0.378119,0.625605,0.535458,0.291909,0.620536
511,CLONE_2520,0.676299,0.346800,8.358086,0.676299,0.978670,0.839521,0.562717,0.732861,0.613445,0.582754,0.389525,0.639222,0.499225,0.321171,0.371455
621,CLONE_3254,0.815263,0.310529,8.071338,0.815263,0.736863,0.767834,0.572193,0.731701,0.466258,0.676087,0.538138,0.475905,0.677919,0.500052,0.400467
473,CLONE_2779,0.635943,0.351138,9.458827,0.635943,0.995446,0.934453,0.551145,0.731421,0.562589,0.725839,0.639595,0.390265,0.461603,0.410577,0.535068
898,CLONE_3393,0.680823,0.376229,8.988781,0.680823,0.895086,0.997195,0.517569,0.730318,0.655951,0.683749,0.397058,0.391964,0.500291,0.516691,0.625119
496,CLONE_0545,0.652604,0.337253,8.433451,0.652604,0.915019,0.858363,0.607788,0.724747,0.759494,0.726823,0.456857,0.254223,0.610395,0.192800,0.251314
803,CLONE_1437,0.670519,0.361212,8.198667,0.670519,0.955153,0.799667,0.578250,0.723751,0.555393,0.572572,0.651557,0.543225,0.623725,0.747091,0.467463


## Step 4 — Build clone × process table

We expand the dataset so that each clone appears once per process condition.

This allows us to evaluate:

> clone A under baseline  
> clone A under rich media  
> clone A under nutrient limitation  
> clone A under balanced feed

The selection problem becomes:

> Which clone-process pair is best?

In [5]:
# --------------------------------------------------
# Step 4 — Expand clones into clone × process pairs
# --------------------------------------------------

rows = []

for process_name, effect in process_conditions.items():
    tmp = work.copy()
    tmp["process_condition"] = process_name
    tmp["base_qp_effect"] = effect["qp_effect"]
    tmp["base_drop_effect"] = effect["drop_effect"]
    tmp["base_agg_effect"] = effect["agg_effect"]
    rows.append(tmp)

pair_df = pd.concat(rows, ignore_index=True)

print("Clone-process table shape:", pair_df.shape)
print("Number of unique clones:", pair_df["clone_id"].nunique())
print("Process conditions:", pair_df["process_condition"].unique())

display(pair_df[[
    "clone_id",
    "process_condition",
    "pred_late_qp",
    "pred_qp_drop",
    "pred_late_agg",
    "process_sensitivity",
]].head(12))

Clone-process table shape: (6000, 38)
Number of unique clones: 1000
Process conditions: <StringArray>
[        'baseline',       'rich_media', 'nutrient_limited',
    'balanced_feed',  'mild_temp_shift',   'perfusion_like']
Length: 6, dtype: str


,clone_id,process_condition,pred_late_qp,pred_qp_drop,pred_late_agg,process_sensitivity
0,CLONE_1502,baseline,2.711177e-08,0.381373,9.583127,0.692021
1,CLONE_2587,baseline,6.445184e-08,0.489493,3.127400,0.348072
2,CLONE_2654,baseline,3.071979e-08,0.362087,7.313653,0.610822
3,CLONE_1056,baseline,6.793292e-08,0.353752,6.868662,0.622844
4,CLONE_0706,baseline,1.729449e-07,0.616517,8.278383,0.409086
5,CLONE_0107,baseline,9.352212e-08,0.376940,5.487992,0.496886
6,CLONE_0590,baseline,3.982962e-07,0.465153,6.215167,0.439847
7,CLONE_2469,baseline,6.033715e-08,0.340156,4.211149,0.482992
8,CLONE_2414,baseline,5.582348e-08,0.399376,4.386003,0.425787
9,CLONE_1601,baseline,1.320460e-07,0.369104,3.793750,0.476660


## Step 5 — Apply clone × process interaction effects

This is the key update of Notebook07.

The previous version applied mostly global process effects.

This version makes process response clone-specific:

- rich media benefits clones with high feed responsiveness, but increases risk in high-burden clones
- nutrient limitation helps high-risk clones reduce stability and aggregation risk
- balanced feed benefits clones with efficient nutrient utilization
- mild temperature shift benefits clones with temperature-shift responsiveness and secretion burden
- perfusion-like process benefits clones with high perfusion rescue potential

The model remains conservative.

Process effects are capped to prevent unrealistic virtual rescue.

In [6]:
# --------------------------------------------------
# Step 5 — Apply clone × process interaction effects
# --------------------------------------------------

# Base response multiplier from overall process sensitivity
pair_df["response_multiplier"] = 0.50 + 0.50 * pair_df["process_sensitivity"]

# Condition flags
is_baseline = pair_df["process_condition"] == "baseline"
is_rich = pair_df["process_condition"] == "rich_media"
is_limited = pair_df["process_condition"] == "nutrient_limited"
is_balanced = pair_df["process_condition"] == "balanced_feed"
is_temp = pair_df["process_condition"] == "mild_temp_shift"
is_perfusion = pair_df["process_condition"] == "perfusion_like"

# Default adjusted effects
pair_df["adj_qp_effect"] = pair_df["base_qp_effect"] * pair_df["response_multiplier"]
pair_df["adj_drop_effect"] = pair_df["base_drop_effect"] * pair_df["response_multiplier"]
pair_df["adj_agg_effect"] = pair_df["base_agg_effect"] * pair_df["response_multiplier"]

# Initialize interaction diagnostics
pair_df["interaction_gain"] = 0.0
pair_df["interaction_risk"] = 0.0
pair_df["interaction_note"] = "baseline/no special interaction"

# -----------------------------
# Rich media
# -----------------------------
# Productivity gain depends on feed responsiveness and nutrient utilization.
# Risk increases with secretion burden and process risk sensitivity.
rich_gain = (
    0.50 * pair_df["feed_responsiveness"]
    + 0.30 * pair_df["nutrient_utilization_efficiency"]
    - 0.25 * pair_df["secretion_burden_index"]
).clip(0, 1)

rich_risk = (
    0.60 * pair_df["secretion_burden_index"]
    + 0.40 * pair_df["process_risk_sensitivity"]
).clip(0, 1)

pair_df.loc[is_rich, "adj_qp_effect"] *= 0.70 + 0.60 * rich_gain[is_rich]
pair_df.loc[is_rich, "adj_drop_effect"] *= 0.70 + 0.60 * rich_risk[is_rich]
pair_df.loc[is_rich, "adj_agg_effect"] *= 0.80 + 0.70 * rich_risk[is_rich]

pair_df.loc[is_rich, "interaction_gain"] = rich_gain[is_rich]
pair_df.loc[is_rich, "interaction_risk"] = rich_risk[is_rich]
pair_df.loc[is_rich, "interaction_note"] = "rich-media response driven by feed responsiveness and burden risk"

# -----------------------------
# Nutrient-limited condition
# -----------------------------
# Helps high process-risk / high aggregation-risk clones reduce risk,
# but productivity gain is not expected.
limited_benefit = (
    0.45 * pair_df["process_risk_sensitivity"]
    + 0.30 * pair_df["secretion_burden_index"]
    + 0.25 * pair_df["sens_agg_band"]
).clip(0, 1)

pair_df.loc[is_limited, "adj_drop_effect"] *= 0.80 + 0.80 * limited_benefit[is_limited]
pair_df.loc[is_limited, "adj_agg_effect"] *= 0.80 + 0.80 * limited_benefit[is_limited]
pair_df.loc[is_limited, "adj_qp_effect"] *= 0.80 + 0.30 * (1.0 - limited_benefit[is_limited])

pair_df.loc[is_limited, "interaction_gain"] = limited_benefit[is_limited]
pair_df.loc[is_limited, "interaction_risk"] = 1.0 - limited_benefit[is_limited]
pair_df.loc[is_limited, "interaction_note"] = "nutrient limitation selected for risk reduction"

# -----------------------------
# Balanced feed
# -----------------------------
# Conservative all-around condition.
balanced_gain = (
    0.45 * pair_df["nutrient_utilization_efficiency"]
    + 0.25 * pair_df["stress_adaptation_capacity"]
    + 0.20 * pair_df["feed_responsiveness"]
    + 0.10 * (1.0 - pair_df["process_risk_sensitivity"])
).clip(0, 1)

pair_df.loc[is_balanced, "adj_qp_effect"] *= 0.80 + 0.50 * balanced_gain[is_balanced]
pair_df.loc[is_balanced, "adj_drop_effect"] *= 0.80 + 0.50 * balanced_gain[is_balanced]
pair_df.loc[is_balanced, "adj_agg_effect"] *= 0.80 + 0.50 * balanced_gain[is_balanced]

pair_df.loc[is_balanced, "interaction_gain"] = balanced_gain[is_balanced]
pair_df.loc[is_balanced, "interaction_risk"] = 1.0 - balanced_gain[is_balanced]
pair_df.loc[is_balanced, "interaction_note"] = "balanced feed selected for moderate productivity and risk control"

# -----------------------------
# Mild temperature shift
# -----------------------------
# Useful for high secretion burden clones if they retain adaptation capacity.
temp_gain = (
    0.45 * pair_df["temperature_shift_responsiveness"]
    + 0.25 * pair_df["stress_adaptation_capacity"]
    + 0.20 * pair_df["secretion_burden_index"]
    + 0.10 * (1.0 - pair_df["process_risk_sensitivity"])
).clip(0, 1)

temp_risk = (
    0.50 * pair_df["process_risk_sensitivity"]
    + 0.30 * (1.0 - pair_df["stress_adaptation_capacity"])
    + 0.20 * pair_df["secretion_burden_index"]
).clip(0, 1)

pair_df.loc[is_temp, "adj_qp_effect"] *= 0.70 + 0.60 * temp_gain[is_temp]
pair_df.loc[is_temp, "adj_drop_effect"] *= 0.80 + 0.70 * temp_gain[is_temp]
pair_df.loc[is_temp, "adj_agg_effect"] *= 0.80 + 0.70 * temp_gain[is_temp]

# If temperature shift risk is high, reduce the benefit slightly.
pair_df.loc[is_temp, "adj_qp_effect"] *= 1.0 - 0.20 * temp_risk[is_temp]

pair_df.loc[is_temp, "interaction_gain"] = temp_gain[is_temp]
pair_df.loc[is_temp, "interaction_risk"] = temp_risk[is_temp]
pair_df.loc[is_temp, "interaction_note"] = "temperature-shift response driven by burden relief and stress adaptation"

# -----------------------------
# Perfusion-like condition
# -----------------------------
# Rescue-oriented condition for clones with perfusion rescue potential.
perfusion_gain = (
    0.50 * pair_df["perfusion_rescue_potential"]
    + 0.20 * pair_df["secretion_burden_index"]
    + 0.20 * pair_df["process_risk_sensitivity"]
    + 0.10 * pair_df["stress_adaptation_capacity"]
).clip(0, 1)

perfusion_risk = (
    0.40 * (1.0 - pair_df["stress_adaptation_capacity"])
    + 0.30 * pair_df["process_risk_sensitivity"]
    + 0.30 * (1.0 - pair_df["perfusion_rescue_potential"])
).clip(0, 1)

pair_df.loc[is_perfusion, "adj_qp_effect"] *= 0.80 + 0.50 * perfusion_gain[is_perfusion]
pair_df.loc[is_perfusion, "adj_drop_effect"] *= 0.80 + 0.80 * perfusion_gain[is_perfusion]
pair_df.loc[is_perfusion, "adj_agg_effect"] *= 0.80 + 0.70 * perfusion_gain[is_perfusion]

# If perfusion risk is high, cap productivity gain.
pair_df.loc[is_perfusion, "adj_qp_effect"] *= 1.0 - 0.15 * perfusion_risk[is_perfusion]

pair_df.loc[is_perfusion, "interaction_gain"] = perfusion_gain[is_perfusion]
pair_df.loc[is_perfusion, "interaction_risk"] = perfusion_risk[is_perfusion]
pair_df.loc[is_perfusion, "interaction_note"] = "perfusion-like rescue driven by rescue potential and burden/waste sensitivity"

# -----------------------------
# Conservative caps
# -----------------------------
pair_df["adj_qp_effect"] = pair_df["adj_qp_effect"].clip(-0.08, 0.12)
pair_df["adj_drop_effect"] = pair_df["adj_drop_effect"].clip(-0.12, 0.06)
pair_df["adj_agg_effect"] = pair_df["adj_agg_effect"].clip(-0.12, 0.08)

# Apply effects
pair_df["process_late_qp"] = (
    pair_df["pred_late_qp"] * (1.0 + pair_df["adj_qp_effect"])
)

pair_df["process_qp_drop"] = (
    pair_df["pred_qp_drop"] + pair_df["adj_drop_effect"]
).clip(lower=0.0)

pair_df["process_late_agg"] = (
    pair_df["pred_late_agg"] * (1.0 + pair_df["adj_agg_effect"])
).clip(lower=0.0)

display(
    pair_df[
        [
            "clone_id",
            "process_condition",
            "process_sensitivity",
            "interaction_gain",
            "interaction_risk",
            "adj_qp_effect",
            "adj_drop_effect",
            "adj_agg_effect",
            "pred_late_qp",
            "process_late_qp",
            "pred_qp_drop",
            "process_qp_drop",
            "pred_late_agg",
            "process_late_agg",
            "interaction_note",
        ]
    ]
    .sort_values(["process_sensitivity", "interaction_gain"], ascending=False)
    .head(25)
)

,clone_id,process_condition,process_sensitivity,interaction_gain,interaction_risk,adj_qp_effect,adj_drop_effect,adj_agg_effect,pred_late_qp,process_late_qp,pred_qp_drop,process_qp_drop,pred_late_agg,process_late_agg,interaction_note
4180,CLONE_4625,mild_temp_shift,0.895472,0.708019,0.354759,0.029712,-0.061395,-0.073674,5.537450e-06,5.701977e-06,0.357910,0.296515,8.834887,8.183985,temperature-shift response driven by burden re...
3180,CLONE_4625,balanced_feed,0.895472,0.673854,0.326146,0.043100,-0.043100,-0.043100,5.537450e-06,5.776116e-06,0.357910,0.314809,8.834887,8.454100,balanced feed selected for moderate productivi...
2180,CLONE_4625,nutrient_limited,0.895472,0.582231,0.417769,-0.035079,-0.095970,-0.095970,5.537450e-06,5.343203e-06,0.357910,0.261939,8.834887,7.986999,nutrient limitation selected for risk reduction
5180,CLONE_4625,perfusion_like,0.895472,0.486456,0.351394,0.018732,-0.067621,-0.054046,5.537450e-06,5.641177e-06,0.357910,0.290289,8.834887,8.357400,perfusion-like rescue driven by rescue potenti...
1180,CLONE_4625,rich_media,0.895472,0.241594,0.572483,0.064064,0.029669,0.068279,5.537450e-06,5.892200e-06,0.357910,0.387578,8.834887,9.438123,rich-media response driven by feed responsiven...
180,CLONE_4625,baseline,0.895472,0.000000,0.000000,0.000000,0.000000,0.000000,5.537450e-06,5.537450e-06,0.357910,0.357910,8.834887,8.834887,baseline/no special interaction
3425,CLONE_4976,balanced_feed,0.751343,0.625103,0.374897,0.038969,-0.038969,-0.038969,7.819973e-08,8.124711e-08,0.349117,0.310148,9.046036,8.693519,balanced feed selected for moderate productivi...
4425,CLONE_4976,mild_temp_shift,0.751343,0.555068,0.270968,0.025667,-0.052039,-0.062447,7.819973e-08,8.020691e-08,0.349117,0.297079,9.046036,8.481141,temperature-shift response driven by burden re...
2425,CLONE_4976,nutrient_limited,0.751343,0.499669,0.500331,-0.033279,-0.084046,-0.084046,7.819973e-08,7.559732e-08,0.349117,0.265072,9.046036,8.285753,nutrient limitation selected for risk reduction
5425,CLONE_4976,perfusion_like,0.751343,0.358780,0.367245,0.016208,-0.057113,-0.046023,7.819973e-08,7.946716e-08,0.349117,0.292005,9.046036,8.629710,perfusion-like rescue driven by rescue potenti...


## Step 6 — Compute clone-process utility

Now each clone-process pair receives a utility score.

The score uses process-adjusted predicted values:

- process_late_qp
- process_qp_drop
- process_late_agg

This score represents expected performance under a specific process condition.

In [7]:
# --------------------------------------------------
# Step 6 — Compute clone-process utility
# --------------------------------------------------

pair_df["raw_process_pair_score"] = make_pred_utility(
    pair_df,
    qp_col="process_late_qp",
    drop_col="process_qp_drop",
    agg_col="process_late_agg",
)

# Add a small risk penalty so high-risk process recommendations
# do not dominate only by predicted productivity gain.
pair_df["process_pair_score"] = (
    pair_df["raw_process_pair_score"]
    - 0.15 * z(pair_df["interaction_risk"])
)

display(
    pair_df[
        [
            "clone_id",
            "process_condition",
            "process_pair_score",
            "raw_process_pair_score",
            "interaction_gain",
            "interaction_risk",
            "process_late_qp",
            "process_qp_drop",
            "process_late_agg",
            "pred_rescue_score",
            "pred_rescue_label",
            "true_util",
        ]
    ]
    .sort_values("process_pair_score", ascending=False)
    .head(15)
)

,clone_id,process_condition,process_pair_score,raw_process_pair_score,interaction_gain,interaction_risk,process_late_qp,process_qp_drop,process_late_agg,pred_rescue_score,pred_rescue_label,true_util
4180,CLONE_4625,mild_temp_shift,13.130287,13.104068,0.708019,0.354759,0.000006,0.296515,8.183985,1.000000,1,3.400230
3180,CLONE_4625,balanced_feed,13.125066,13.079197,0.673854,0.326146,0.000006,0.314809,8.454100,1.000000,1,3.400230
5180,CLONE_4625,perfusion_like,13.038466,13.009936,0.486456,0.351394,0.000006,0.290289,8.357400,1.000000,1,3.400230
2180,CLONE_4625,nutrient_limited,12.620412,12.637463,0.582231,0.417769,0.000005,0.261939,7.986999,1.000000,1,3.400230
1180,CLONE_4625,rich_media,12.467773,12.591072,0.241594,0.572483,0.000006,0.387578,9.438123,1.000000,1,3.400230
180,CLONE_4625,baseline,12.394941,12.125097,0.000000,0.000000,0.000006,0.357910,8.834887,1.000000,1,3.400230
5621,CLONE_3254,perfusion_like,10.671844,10.668633,0.516782,0.388264,0.000004,0.247491,7.665390,0.815263,1,5.086927
4621,CLONE_3254,mild_temp_shift,10.634969,10.638045,0.543143,0.397418,0.000004,0.259436,7.576463,0.815263,1,5.086927
3621,CLONE_3254,balanced_feed,10.602675,10.625122,0.574375,0.425625,0.000004,0.272876,7.767422,0.815263,1,5.086927
2621,CLONE_3254,nutrient_limited,10.349468,10.407756,0.522184,0.477816,0.000004,0.226178,7.390512,0.815263,1,5.086927


## Step 7 — Select best process condition per clone

For each clone, we select the process condition with the highest process-pair score.

This converts the problem from:

> clone × process pair ranking

back to:

> one best expected process condition per clone

In [8]:
# --------------------------------------------------
# Step 7 — Best process per clone
# --------------------------------------------------

best_pair_per_clone = (
    pair_df.sort_values("process_pair_score", ascending=False)
    .groupby("clone_id", as_index=False)
    .head(1)
    .reset_index(drop=True)
)

print("Best-pair table shape:", best_pair_per_clone.shape)

print("\n=== Best process condition distribution ===")
display(best_pair_per_clone["process_condition"].value_counts())

display(best_pair_per_clone[[
    "clone_id",
    "process_condition",
    "process_pair_score",
    "baseline_score",
    "process_sensitivity",
    "pred_rescue_score",
    "pred_rescue_label",
    "process_late_qp",
    "process_qp_drop",
    "process_late_agg",
    "true_late_qp",
    "true_qp_drop",
    "true_late_agg",
    "true_util",
]].sort_values("process_pair_score", ascending=False).head(15))

Best-pair table shape: (1000, 50)

=== Best process condition distribution ===


process_condition
nutrient_limited    617
perfusion_like      192
mild_temp_shift     181
balanced_feed         9
baseline              1
Name: count, dtype: int64

,clone_id,process_condition,process_pair_score,baseline_score,process_sensitivity,pred_rescue_score,pred_rescue_label,process_late_qp,process_qp_drop,process_late_agg,true_late_qp,true_qp_drop,true_late_agg,true_util
0,CLONE_4625,mild_temp_shift,13.130287,12.655775,0.895472,1.000000,1,5.701977e-06,0.296515,8.183985,1.017815e-05,0.440572,11.935039,3.400230
1,CLONE_3254,perfusion_like,10.671844,10.401446,0.731701,0.815263,1,4.385223e-06,0.247491,7.665390,8.830813e-06,0.000000,9.991042,5.086927
2,CLONE_2776,balanced_feed,10.064043,9.697306,0.463818,0.583052,0,5.652462e-06,0.635405,6.941593,7.851137e-05,0.742645,3.219493,28.723011
3,CLONE_0048,balanced_feed,9.871583,9.519451,0.476625,0.585009,1,5.511670e-06,0.585661,10.681668,1.974793e-05,0.563132,11.626145,6.511386
4,CLONE_4878,balanced_feed,9.482234,9.111578,0.708152,0.776958,1,4.589041e-06,0.413460,9.746288,4.980082e-06,0.663904,5.051639,0.764343
5,CLONE_2171,balanced_feed,8.719475,8.301829,0.533623,0.629844,1,5.278185e-06,0.672186,9.048104,5.684738e-06,0.522193,8.604411,1.490784
6,CLONE_1619,perfusion_like,6.822136,6.677909,0.354758,0.294760,0,2.256251e-06,0.204269,2.975840,2.551172e-06,0.107071,0.670928,2.733914
7,CLONE_3863,nutrient_limited,6.434084,6.097181,0.503599,0.615623,1,4.082230e-06,0.648851,7.725760,2.970790e-06,0.715311,3.496560,-0.156112
8,CLONE_0643,nutrient_limited,3.003364,2.708040,0.389805,0.432967,0,1.734664e-06,0.461851,5.884907,1.814575e-06,0.512626,6.048606,0.205427
9,CLONE_2759,nutrient_limited,2.974947,2.502396,0.525781,0.448798,0,7.871062e-07,0.199962,9.951908,1.384358e-06,0.000000,12.804618,2.058771


## Step 8 — Compare baseline clone selection vs clone-process selection

We compare:

1. baseline Top10 selected by original predicted utility
2. clone-process Top10 selected by best process-pair score

This tells us whether process pairing changes clone selection.

In [9]:
# --------------------------------------------------
# Step 8 — Top10 comparison
# --------------------------------------------------

TOP_K = 10

baseline_top10 = (
    work.sort_values("baseline_score", ascending=False)
    .head(TOP_K)
    .copy()
)

process_pair_top10 = (
    best_pair_per_clone.sort_values("process_pair_score", ascending=False)
    .head(TOP_K)
    .copy()
)

baseline_ids = set(baseline_top10["clone_id"])
process_ids = set(process_pair_top10["clone_id"])

print("=== Top10 comparison ===")
print("Baseline rescue count:", int(baseline_top10["pred_rescue_label"].sum()))
print("Process-pair rescue count:", int(process_pair_top10["pred_rescue_label"].sum()))
print(f"Top10 overlap: {len(baseline_ids & process_ids)}/{TOP_K}")

print("\n=== Baseline Top10 ===")
display(baseline_top10[[
    "clone_id",
    "baseline_score",
    "pred_rescue_score",
    "pred_rescue_label",
    "pred_late_qp",
    "pred_qp_drop",
    "pred_late_agg",
    "true_late_qp",
    "true_qp_drop",
    "true_late_agg",
    "true_util",
]])

print("\n=== Process-pair Top10 ===")
display(process_pair_top10[[
    "clone_id",
    "process_condition",
    "process_pair_score",
    "baseline_score",
    "process_sensitivity",
    "pred_rescue_score",
    "pred_rescue_label",
    "process_late_qp",
    "process_qp_drop",
    "process_late_agg",
    "true_late_qp",
    "true_qp_drop",
    "true_late_agg",
    "true_util",
]])

=== Top10 comparison ===
Baseline rescue count: 6
Process-pair rescue count: 6
Top10 overlap: 10/10

=== Baseline Top10 ===


,clone_id,baseline_score,pred_rescue_score,pred_rescue_label,pred_late_qp,pred_qp_drop,pred_late_agg,true_late_qp,true_qp_drop,true_late_agg,true_util
180,CLONE_4625,12.655775,1.000000,1,5.537450e-06,0.357910,8.834887,0.000010,0.440572,11.935039,3.400230
621,CLONE_3254,10.401446,0.815263,1,4.310815e-06,0.310529,8.071338,0.000009,0.000000,9.991042,5.086927
986,CLONE_2776,9.697306,0.583052,0,5.472555e-06,0.668280,7.177550,0.000079,0.742645,3.219493,28.723011
505,CLONE_0048,9.519451,0.585009,1,5.334487e-06,0.618876,11.048644,0.000020,0.563132,11.626145,6.511386
730,CLONE_4878,9.111578,0.776958,1,4.417263e-06,0.452348,10.140637,0.000005,0.663904,5.051639,0.764343
524,CLONE_2171,8.301829,0.629844,1,5.101682e-06,0.706783,9.372360,0.000006,0.522193,8.604411,1.490784
314,CLONE_1619,6.677909,0.294760,0,2.226262e-06,0.252291,3.094684,0.000003,0.107071,0.670928,2.733914
63,CLONE_3863,6.097181,0.615623,1,4.193785e-06,0.731521,8.422007,0.000003,0.715311,3.496560,-0.156112
23,CLONE_0643,2.708040,0.432967,0,1.779974e-06,0.533633,6.340006,0.000002,0.512626,6.048606,0.205427
476,CLONE_2759,2.502396,0.448798,0,8.089825e-07,0.283589,10.860110,0.000001,0.000000,12.804618,2.058771



=== Process-pair Top10 ===


,clone_id,process_condition,process_pair_score,baseline_score,process_sensitivity,pred_rescue_score,pred_rescue_label,process_late_qp,process_qp_drop,process_late_agg,true_late_qp,true_qp_drop,true_late_agg,true_util
0,CLONE_4625,mild_temp_shift,13.130287,12.655775,0.895472,1.000000,1,5.701977e-06,0.296515,8.183985,0.000010,0.440572,11.935039,3.400230
1,CLONE_3254,perfusion_like,10.671844,10.401446,0.731701,0.815263,1,4.385223e-06,0.247491,7.665390,0.000009,0.000000,9.991042,5.086927
2,CLONE_2776,balanced_feed,10.064043,9.697306,0.463818,0.583052,0,5.652462e-06,0.635405,6.941593,0.000079,0.742645,3.219493,28.723011
3,CLONE_0048,balanced_feed,9.871583,9.519451,0.476625,0.585009,1,5.511670e-06,0.585661,10.681668,0.000020,0.563132,11.626145,6.511386
4,CLONE_4878,balanced_feed,9.482234,9.111578,0.708152,0.776958,1,4.589041e-06,0.413460,9.746288,0.000005,0.663904,5.051639,0.764343
5,CLONE_2171,balanced_feed,8.719475,8.301829,0.533623,0.629844,1,5.278185e-06,0.672186,9.048104,0.000006,0.522193,8.604411,1.490784
6,CLONE_1619,perfusion_like,6.822136,6.677909,0.354758,0.294760,0,2.256251e-06,0.204269,2.975840,0.000003,0.107071,0.670928,2.733914
7,CLONE_3863,nutrient_limited,6.434084,6.097181,0.503599,0.615623,1,4.082230e-06,0.648851,7.725760,0.000003,0.715311,3.496560,-0.156112
8,CLONE_0643,nutrient_limited,3.003364,2.708040,0.389805,0.432967,0,1.734664e-06,0.461851,5.884907,0.000002,0.512626,6.048606,0.205427
9,CLONE_2759,nutrient_limited,2.974947,2.502396,0.525781,0.448798,0,7.871062e-07,0.199962,9.951908,0.000001,0.000000,12.804618,2.058771


## Step 9 — Utility overlap and true performance

We evaluate whether clone-process selection better recovers truly good clones.

Utility overlap compares:

- true Top10 by true_util
- baseline selected Top10
- process-pair selected Top10

We also compare true performance averages.

In [10]:
# --------------------------------------------------
# Step 9 — Utility overlap and true performance
# --------------------------------------------------

true_top10 = work.sort_values("true_util", ascending=False).head(TOP_K)
true_top_ids = set(true_top10["clone_id"])

baseline_overlap = len(true_top_ids & set(baseline_top10["clone_id"])) / TOP_K
process_pair_overlap = len(true_top_ids & set(process_pair_top10["clone_id"])) / TOP_K

print("=== Utility overlap ===")
print(f"Baseline utility overlap:     {baseline_overlap:.3f}")
print(f"Process-pair utility overlap: {process_pair_overlap:.3f}")

summary = pd.DataFrame({
    "baseline_top10": baseline_top10[[
        "true_late_qp",
        "true_qp_drop",
        "true_late_agg",
        "true_util",
    ]].mean(),
    "process_pair_top10": process_pair_top10[[
        "true_late_qp",
        "true_qp_drop",
        "true_late_agg",
        "true_util",
    ]].mean(),
})

summary["direction"] = [
    "higher is better",
    "lower is better",
    "lower is better",
    "higher is better",
]

display(summary)

=== Utility overlap ===
Baseline utility overlap:     0.500
Process-pair utility overlap: 0.500


,baseline_top10,process_pair_top10,direction
true_late_qp,0.000014,0.000014,higher is better
true_qp_drop,0.426745,0.426745,lower is better
true_late_agg,7.344848,7.344848,lower is better
true_util,5.081868,5.081868,higher is better


## Step 10 — Process condition diagnostics

This step checks which process conditions are selected among top candidates.

This helps interpret whether the process model behaves reasonably.

Expected behavior:

- rich_media may appear for productivity-driven clones
- nutrient_limited may appear for unstable or aggregation-prone clones
- balanced_feed should often be a safe compromise

In [11]:
# --------------------------------------------------
# Step 10 — Process condition diagnostics
# --------------------------------------------------

print("=== Process condition distribution in Top10 ===")
display(process_pair_top10["process_condition"].value_counts())

print("\n=== Mean predicted process-adjusted outcomes by selected process ===")
display(process_pair_top10.groupby("process_condition")[[
    "process_late_qp",
    "process_qp_drop",
    "process_late_agg",
    "process_pair_score",
    "true_late_qp",
    "true_qp_drop",
    "true_late_agg",
    "true_util",
]].mean())

print("\n=== Rescue label distribution by selected process ===")
display(pd.crosstab(
    process_pair_top10["process_condition"],
    process_pair_top10["pred_rescue_label"],
    rownames=["process_condition"],
    colnames=["pred_rescue_label"],
))

=== Process condition distribution in Top10 ===


process_condition
balanced_feed       4
nutrient_limited    3
perfusion_like      2
mild_temp_shift     1
Name: count, dtype: int64


=== Mean predicted process-adjusted outcomes by selected process ===


,process_late_qp,process_qp_drop,process_late_agg,process_pair_score,true_late_qp,true_qp_drop,true_late_agg,true_util
process_condition,,,,,,,,
balanced_feed,0.000005,0.576678,9.104413,9.534334,0.000027,0.622968,7.125422,9.372381
mild_temp_shift,0.000006,0.296515,8.183985,13.130287,0.000010,0.440572,11.935039,3.400230
nutrient_limited,0.000002,0.436888,7.854192,4.137465,0.000002,0.409312,7.449928,0.702695
perfusion_like,0.000003,0.225880,5.320615,8.746990,0.000006,0.053535,5.330985,3.910420



=== Rescue label distribution by selected process ===


pred_rescue_label,0,1
process_condition,,
balanced_feed,1,3
mild_temp_shift,0,1
nutrient_limited,2,1
perfusion_like,1,1


## Step 11 — Production Top3 guardrail

Top10 can include exploratory clone-process pairs.

However, Top3 should remain conservative because these represent production-like candidates.

Here we select Top3 from process-pair Top10 but require:

- acceptable predicted qP drop
- acceptable predicted aggregation

In [12]:
# --------------------------------------------------
# Step 11 — Conservative Top3 production candidates
# --------------------------------------------------

TOP3_MAX_DROP = 0.45
TOP3_MAX_AGG = 14.0

top3_pool = process_pair_top10[
    (process_pair_top10["process_qp_drop"] <= TOP3_MAX_DROP) &
    (process_pair_top10["process_late_agg"] <= TOP3_MAX_AGG)
].copy()

process_pair_top3 = (
    top3_pool.sort_values("process_pair_score", ascending=False)
    .head(3)
    .copy()
)

print("Top3 candidates:", len(process_pair_top3))

display(process_pair_top3[[
    "clone_id",
    "process_condition",
    "process_pair_score",
    "process_late_qp",
    "process_qp_drop",
    "process_late_agg",
    "pred_rescue_score",
    "pred_rescue_label",
    "true_late_qp",
    "true_qp_drop",
    "true_late_agg",
    "true_util",
]])

Top3 candidates: 3


,clone_id,process_condition,process_pair_score,process_late_qp,process_qp_drop,process_late_agg,pred_rescue_score,pred_rescue_label,true_late_qp,true_qp_drop,true_late_agg,true_util
0,CLONE_4625,mild_temp_shift,13.130287,0.000006,0.296515,8.183985,1.000000,1,0.000010,0.440572,11.935039,3.400230
1,CLONE_3254,perfusion_like,10.671844,0.000004,0.247491,7.665390,0.815263,1,0.000009,0.000000,9.991042,5.086927
4,CLONE_4878,balanced_feed,9.482234,0.000005,0.413460,9.746288,0.776958,1,0.000005,0.663904,5.051639,0.764343


## Step 12 — Action recommendation table

The previous steps selected clone × process pairs.

This step converts those results into a practical decision-support output.

Instead of only reporting scores, we generate an action table with:

- clone ID
- selected process condition
- recommended action
- expected performance after process pairing
- expected improvement versus baseline
- rescue / pass status
- confidence tier
- biological rationale

This is the table a CLD or CDMO scientist would actually review before deciding which clones should move forward, which clones should receive process optimization, and which clones should remain exploratory.

In [13]:
# --------------------------------------------------
# Step 12 — Action recommendation table
# --------------------------------------------------

def find_existing_df(candidate_names):
    for name in candidate_names:
        if name in globals() and isinstance(globals()[name], pd.DataFrame):
            print(f"Using dataframe: {name}")
            return globals()[name].copy(), name

    raise NameError(
        f"None of these dataframes were found: {candidate_names}. "
        "Run the previous Notebook07 selection cells first."
    )

process_top10_df, _ = find_existing_df([
    "process_pair_top10",
    "process_top10",
    "top10_process",
    "top10_pairs",
    "top10",
])

process_top3_df, _ = find_existing_df([
    "process_pair_top3",
    "production_top3",
    "process_top3",
    "top3_process",
    "top3_pairs",
    "top3",
])

action = process_top10_df.copy()
top3_ids = set(process_top3_df["clone_id"])

# Expected improvement vs baseline
action["delta_late_qp"] = action["process_late_qp"] - action["pred_late_qp"]
action["delta_late_qp_pct"] = action["delta_late_qp"] / (
    action["pred_late_qp"].abs() + 1e-12
)

action["delta_qp_drop"] = action["process_qp_drop"] - action["pred_qp_drop"]
action["delta_late_agg"] = action["process_late_agg"] - action["pred_late_agg"]

# Decision role
action["decision_role"] = np.where(
    action["clone_id"].isin(top3_ids),
    "Top3 production candidate",
    "Top10 process-development candidate",
)

def recommended_action(row):
    rescue = int(row.get("pred_rescue_label", 0))
    process = row["process_condition"]
    drop = row["process_qp_drop"]
    agg = row["process_late_agg"]
    risk = row.get("interaction_risk", 0.0)

    if row["decision_role"] == "Top3 production candidate":
        if rescue == 1:
            return "Advance cautiously as production candidate with process-confirmation run"
        return "Advance as production candidate"

    if process == "perfusion_like" and rescue == 1:
        return "Process-rescue candidate: test perfusion-like media-exchange strategy"

    if process == "mild_temp_shift" and rescue == 1:
        return "Process-rescue candidate: test mild temperature-shift strategy"

    if process == "balanced_feed" and rescue == 1:
        return "Process-rescue candidate: test balanced-feed optimization"

    if process == "nutrient_limited":
        return "Risk-reduction candidate: test nutrient-limited condition"

    if process == "rich_media":
        if risk > 0.65:
            return "Exploratory rich-media candidate with strong risk monitoring"
        return "Productivity-boost candidate: test rich-media condition"

    if drop > 0.45:
        return "Monitor stability risk before advancement"

    if agg > 14.0:
        return "Monitor aggregation risk before advancement"

    return "Process-development candidate"

def confidence_tier(row):
    rescue = int(row.get("pred_rescue_label", 0))
    stable_prob = row.get("pred_stable_prob", np.nan)
    drop = row["process_qp_drop"]
    agg = row["process_late_agg"]
    gain = row.get("interaction_gain", 0.0)
    risk = row.get("interaction_risk", 0.0)

    if rescue == 0 and stable_prob >= 0.65 and drop <= 0.35 and agg <= 10 and risk <= 0.45:
        return "High"

    if rescue == 0 and stable_prob >= 0.50 and drop <= 0.45 and agg <= 14 and risk <= 0.60:
        return "Medium"

    if rescue == 1 and drop <= 0.45 and agg <= 14 and gain >= 0.45 and risk <= 0.65:
        return "Medium-rescue"

    if rescue == 1 and gain >= 0.55:
        return "Exploratory-rescue"

    return "Low / exploratory"

def rationale(row):
    parts = []

    if int(row.get("pred_rescue_label", 0)) == 1:
        parts.append("rescue-labeled clone with process optimization potential")
    else:
        parts.append("non-rescue clone selected by process-aware ranking")

    process = row["process_condition"]

    if process == "balanced_feed":
        parts.append("balanced-feed selected for nutrient-efficient risk-balanced improvement")
    elif process == "nutrient_limited":
        parts.append("nutrient-limited condition selected for stability/quality risk reduction")
    elif process == "rich_media":
        parts.append("rich-media condition selected for productivity boost")
    elif process == "mild_temp_shift":
        parts.append("mild temperature shift selected for secretion-burden/stress adaptation response")
    elif process == "perfusion_like":
        parts.append("perfusion-like condition selected for rescue/waste-removal potential")
    elif process == "baseline":
        parts.append("baseline remains optimal")

    if row.get("interaction_gain", 0.0) >= 0.55:
        parts.append("high predicted clone-process interaction gain")

    if row.get("interaction_risk", 0.0) >= 0.65:
        parts.append("requires risk monitoring due to process sensitivity")

    if row["delta_late_qp_pct"] > 0.02:
        parts.append("expected qP improvement")

    if row["delta_qp_drop"] < -0.02:
        parts.append("expected qP-drop reduction")

    if row["delta_late_agg"] < -0.2:
        parts.append("expected aggregation reduction")

    return "; ".join(parts)

action["recommended_action"] = action.apply(recommended_action, axis=1)
action["confidence_tier"] = action.apply(confidence_tier, axis=1)
action["rationale"] = action.apply(rationale, axis=1)

action_cols = [
    "clone_id",
    "decision_role",
    "process_condition",
    "recommended_action",
    "confidence_tier",
    "process_pair_score",
    "raw_process_pair_score",
    "interaction_gain",
    "interaction_risk",
    "pred_rescue_label",
    "pred_rescue_score",
    "pred_stable_prob",
    "process_late_qp",
    "process_qp_drop",
    "process_late_agg",
    "delta_late_qp_pct",
    "delta_qp_drop",
    "delta_late_agg",
    "nutrient_utilization_efficiency",
    "stress_adaptation_capacity",
    "perfusion_rescue_potential",
    "temperature_shift_responsiveness",
    "feed_responsiveness",
    "secretion_burden_index",
    "process_risk_sensitivity",
    "rationale",
]

action_cols = [c for c in action_cols if c in action.columns]

action_recommendation_table = (
    action[action_cols]
    .sort_values(
        ["decision_role", "process_pair_score"],
        ascending=[True, False],
    )
    .reset_index(drop=True)
)

print("=== Action recommendation table ===")
print("Shape:", action_recommendation_table.shape)

display(action_recommendation_table)

Using dataframe: process_pair_top10
Using dataframe: process_pair_top3
=== Action recommendation table ===
Shape: (10, 26)


,clone_id,decision_role,process_condition,recommended_action,confidence_tier,process_pair_score,raw_process_pair_score,interaction_gain,interaction_risk,pred_rescue_label,...,delta_qp_drop,delta_late_agg,nutrient_utilization_efficiency,stress_adaptation_capacity,perfusion_rescue_potential,temperature_shift_responsiveness,feed_responsiveness,secretion_burden_index,process_risk_sensitivity,rationale
0,CLONE_2776,Top10 process-development candidate,balanced_feed,Monitor stability risk before advancement,Low / exploratory,10.064043,10.037442,0.645797,0.354203,0,...,-0.032874,-0.235958,0.710317,0.742990,0.396166,0.726733,0.492997,0.744872,0.581925,non-rescue clone selected by process-aware ran...
1,CLONE_0048,Top10 process-development candidate,balanced_feed,Process-rescue candidate: test balanced-feed o...,Exploratory-rescue,9.871583,9.842538,0.649357,0.350643,1,...,-0.033215,-0.366976,0.664708,0.633712,0.328530,0.365677,0.702977,0.802494,0.487854,rescue-labeled clone with process optimization...
2,CLONE_2171,Top10 process-development candidate,balanced_feed,Process-rescue candidate: test balanced-feed o...,Exploratory-rescue,8.719475,8.685934,0.655902,0.344098,1,...,-0.034597,-0.324256,0.602791,0.790550,0.472431,0.229402,0.705316,0.685503,0.540549,rescue-labeled clone with process optimization...
3,CLONE_1619,Top10 process-development candidate,perfusion_like,Process-development candidate,High,6.822136,6.746940,0.476944,0.283441,0,...,-0.048022,-0.118844,0.651338,0.996386,0.419988,0.535722,0.664161,0.476582,0.359974,non-rescue clone selected by process-aware ran...
4,CLONE_3863,Top10 process-development candidate,nutrient_limited,Risk-reduction candidate: test nutrient-limite...,Exploratory-rescue,6.434084,6.357783,0.718168,0.281832,1,...,-0.082670,-0.696247,0.575114,0.342411,0.408838,0.661281,0.729435,0.807576,0.582267,rescue-labeled clone with process optimization...
5,CLONE_0643,Top10 process-development candidate,nutrient_limited,Risk-reduction candidate: test nutrient-limite...,Low / exploratory,3.003364,2.998577,0.614033,0.385967,0,...,-0.071782,-0.455099,0.363718,0.621744,0.500147,0.437355,0.358741,0.883324,0.589523,non-rescue clone selected by process-aware ran...
6,CLONE_2759,Top10 process-development candidate,nutrient_limited,Risk-reduction candidate: test nutrient-limite...,Medium,2.974947,2.902334,0.712797,0.287203,0,...,-0.083627,-0.908202,0.434319,0.740824,0.522102,0.642478,0.324470,0.667000,0.731399,non-rescue clone selected by process-aware ran...
7,CLONE_4625,Top3 production candidate,mild_temp_shift,Advance cautiously as production candidate wit...,Medium-rescue,13.130287,13.104068,0.708019,0.354759,1,...,-0.061395,-0.650901,0.718573,0.728852,0.415454,0.638964,0.454035,0.803980,0.225237,rescue-labeled clone with process optimization...
8,CLONE_3254,Top3 production candidate,perfusion_like,Advance cautiously as production candidate wit...,Medium-rescue,10.671844,10.668633,0.516782,0.388264,1,...,-0.063039,-0.405948,0.466258,0.676087,0.538138,0.475905,0.677919,0.500052,0.400467,rescue-labeled clone with process optimization...
9,CLONE_4878,Top3 production candidate,balanced_feed,Advance cautiously as production candidate wit...,Medium-rescue,9.482234,9.434471,0.676613,0.323387,1,...,-0.038888,-0.394349,0.694753,0.749096,0.393284,0.592410,0.579437,0.603617,0.391874,rescue-labeled clone with process optimization...


### Interpretation — action recommendation table

This table is the practical output of Notebook07.

The key transition is:

> clone ranking → clone × process action recommendation

Each row now tells us:

- which clone to consider
- which process condition is recommended
- whether the clone is a direct production candidate or process-development candidate
- whether it is a rescue clone
- what improvement is expected versus baseline
- why the recommendation was made

This is still a simulation output.  
The process effects are not yet learned from real media/feed/process data.

However, this table is the first version of a CDMO-facing decision output.

In [14]:
# --------------------------------------------------
# Save action recommendation table
# --------------------------------------------------

if "action_recommendation_table" not in globals():
    raise NameError(
        "action_recommendation_table is not defined. "
        "Run Step 12 — Action recommendation table first."
    )

OUT_ACTION = Path("../reports") / f"notebook07_action_recommendations_{n_clones}_{scenario}.csv"
OUT_ACTION.parent.mkdir(parents=True, exist_ok=True)

action_recommendation_table.to_csv(OUT_ACTION, index=False)

print("Saved action recommendation table:", OUT_ACTION)
print("Shape:", action_recommendation_table.shape)

Saved action recommendation table: ../reports/notebook07_action_recommendations_5000_legacy.csv
Shape: (10, 26)


## Final interpretation of Notebook07

Notebook07 extends the CLD pipeline from clone-only decision-making to **clone × process interaction decision-making**.

The main output is no longer only a ranked clone list.

It is an action recommendation table:

> clone ID → recommended process condition → predicted post-process profile → interaction gain/risk → decision role → rationale

---

## What changed in this version

Earlier Notebook07 versions used mostly global process effects.

This version uses generator-derived clone-specific process-response latents:

- nutrient utilization efficiency
- stress adaptation capacity
- perfusion rescue potential
- temperature shift responsiveness
- feed responsiveness
- secretion burden index
- process risk sensitivity

These variables allow the same process condition to affect different clones differently.

This is the key transition:

> static clone ranking → clone-specific process response simulation

---

## What this notebook demonstrates

1. Clone ranking can change under different process conditions.
2. Process response is clone-specific rather than uniform.
3. Rescue clones may become competitive only under selected process conditions.
4. Some clones benefit from productivity-oriented conditions such as rich media.
5. Some clones benefit more from risk-reduction strategies such as nutrient limitation, mild temperature shift, or perfusion-like rescue.
6. Top10 can be used as an exploratory clone-process candidate pool.
7. Top3 remains protected by conservative production guardrails.
8. The final action table translates model outputs into scientist-readable recommendations.

---

## Practical output

The action recommendation table is the key output of this notebook.

It summarizes:

- clone_id
- decision_role
- process_condition
- recommended_action
- confidence_tier
- predicted post-process qP, qP drop, and aggregation
- expected change versus baseline
- interaction gain
- interaction risk
- process-response latent profile
- biological / process rationale

This makes Notebook07 closer to a CDMO-facing decision-support layer.

---

## Scientific interpretation

This notebook should be interpreted as a **hypothesis-generating clone × process simulation framework**.

It does not claim that the simulated process effects are experimentally validated.

Instead, it tests whether the pipeline can represent a realistic CLD/PD idea:

> some clones should be dropped early,  
> some clones can advance directly,  
> and some borderline clones may only become valuable under the right process condition.

---

## Current limitation

This notebook is still simulation-based.

The process effects are not learned from real:

- LC/GC spent media data
- Raman / PAT measurements
- media composition
- feed schedule
- perfusion bioreactor data
- glycosylation profiles
- scale-up runs

Therefore, the results should not be interpreted as validated process recommendations yet.

They should be interpreted as:

> structured hypotheses for which clone × process experiments should be tested next.

---

## Next direction

The next major improvement is to replace or calibrate these latent process-response assumptions with real data.

Possible future extensions:

1. Add glycosylation coupling.
2. Add media composition variables.
3. Add spent-media metabolite features.
4. Add perfusion vs fed-batch process response data.
5. Add omics-derived latent states such as secretion stress, UPR, metabolic burden, and epigenetic drift.

This version establishes the architecture needed for that future work.